In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedShuffleSplit
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors

In [ ]:
# ====== 步骤三：载入手写体数字识别数据集并进行标准化 ======
def load_and_standardize_data(file_path):
    """
    载入MAT格式的手写数字数据集并进行标准化
    
    参数:
    file_path -- MAT文件路径
    
    返回:
    X -- 标准化后的特征矩阵 (5000, 400)
    y -- 标签向量 (5000,)
    """
    # 加载数据集
    data = loadmat(file_path)
    X = data['X']  # 特征矩阵 (5000, 400)
    y = data['y'].ravel()  # 标签向量 (5000,)
    y[y == 10] = 0  # 将标签10改为0
    
    # 标准化数据：使每个特征服从标准正态分布
    scaler = StandardScaler()
    X_standardized = scaler.fit_transform(X)
    
    return X_standardized, y

In [ ]:
# 1. 加载并标准化数据
data_path = "/mnt/e/平时作业/机器学习/实验1/ex3data1.mat"  # 替换为您的实际文件路径
X, y = load_and_standardize_data(data_path)
    
print("原始数据信息:")
print(f"样本数: {X.shape[0]}, 特征数: {X.shape[1]}")

In [ ]:
# ====== 步骤四：手动实现PCA函数 ======
def manual_pca(X, n_components=2):
    """
    手动实现主成分分析(PCA)
    
    参数:
    X -- 输入数据矩阵 (m, n)，其中m是样本数，n是特征数
    n_components -- 要保留的主成分数量
    
    返回:
    X_pca -- 降维后的数据矩阵 (m, n_components)
    explained_variance_ratio -- 各主成分解释的方差比例
    """
    # 1. 计算协方差矩阵
    cov_matrix = np.cov(X.T)
    
    # 2. 计算特征值和特征向量
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # 3. 按特征值大小降序排列
    sorted_index = np.argsort(eigenvalues)[::-1]
    sorted_eigenvectors = eigenvectors[:, sorted_index]
    
    # 4. 选择前n_components个特征向量
    top_eigenvectors = sorted_eigenvectors[:, :n_components]
    
    # 5. 计算主成分并返回
    X_pca = X @ top_eigenvectors
    
    # 计算解释的方差比例
    total_variance = eigenvalues.sum()
    explained_variance_ratio = eigenvalues[sorted_index][:n_components] / total_variance
    
    return X_pca, explained_variance_ratio

In [ ]:
# ====== 步骤五：可视化降维结果 ======
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 用于3D绘图

def visualize_pca_results(X_pca, y, n_components):
    """
    可视化PCA降维结果
    
    参数:
    X_pca : 降维后的数据 (m x n_components)
    y : 标签向量 (m,)
    n_components : 降维后的维度 (2或3)
    """
    plt.figure(figsize=(12, 8))
    
    if n_components == 2:
        # 二维可视化
        scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='tab10', s=10)
        plt.title('PCA 2D Visualization of Handwritten Digits')
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
        plt.colorbar(scatter, label='Digit Class')
        
    elif n_components == 3:
        # 三维可视化
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=y, cmap='tab10', s=10)
        
        ax.set_title('PCA 3D Visualization of Handwritten Digits')
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3')
        fig.colorbar(scatter, label='Digit Class', pad=0.1)
        
        # 设置视角（可选）
        ax.view_init(elev=20, azim=35)
    
    plt.tight_layout()
    plt.show()


In [ ]:
# 2. 执行PCA降维 - 手动实现
# 2维降维
print("\n使用手动PCA进行2维降维...")
X_pca_manual_2d, variance_ratio_2d = manual_pca(X, n_components=2)
print(f"解释方差比例: {variance_ratio_2d}")
print(f"总解释方差比例: {variance_ratio_2d.sum():.4f}")
    
# 3维降维
print("\n使用手动PCA进行3维降维...")
X_pca_manual_3d, variance_ratio_3d = manual_pca(X, n_components=3)
print(f"解释方差比例: {variance_ratio_3d}")
print(f"总解释方差比例: {variance_ratio_3d.sum():.4f}")

In [ ]:
# ====== 可视化降维结果 ======
print("\n可视化降维结果...")
# 2D可视化
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca_manual_2d[:, 0], X_pca_manual_2d[:, 1], 
                     c=y, cmap='tab10', s=15)
plt.title('Handwritten Digits Dataset - PCA 2D Projection')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.colorbar(scatter, label='Digit Class')
plt.grid(True, alpha=0.3)
plt.show()

# 3D可视化
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(X_pca_manual_3d[:, 0], X_pca_manual_3d[:, 1], X_pca_manual_3d[:, 2],
                     c=y, cmap='tab10', s=15)

ax.set_title('Handwritten Digits Dataset - PCA 3D Projection')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
fig.colorbar(scatter, label='Digit Class', pad=0.1)
ax.view_init(elev=20, azim=35)  # 设置视角
plt.show()

In [ ]:
# ====== 步骤六：K-Means聚类实现 ======
def kmeans_clustering(X, n_clusters=10, max_iter=300):
    """
    K-Means聚类算法实现
    
    参数:
    X -- 输入数据 (m, n)
    n_clusters -- 聚类数量 (默认10)
    max_iter -- 最大迭代次数 (默认300)
    
    返回:
    labels -- 聚类标签 (m,)
    centroids -- 聚类中心点 (n_clusters, n)
    """
    # 随机初始化中心点
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], n_clusters, replace=False), :]
    
    for _ in range(max_iter):
        # 计算每个点到各中心点的距离
        distances = np.sqrt(((X - centroids[:, np.newaxis])**2).sum(axis=2))
        
        # 分配每个点到最近的中心点
        labels = np.argmin(distances, axis=0)
        
        # 更新中心点位置
        new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(n_clusters)])
        
        # 检查是否收敛
        if np.all(centroids == new_centroids):
            break
            
        centroids = new_centroids
        
    return labels, centroids

In [ ]:
# ====== 步骤六/七/八：可视化函数 ======
def plot_cluster_results(X, labels, true_labels=None, dimension=2, title=""):
    """
    可视化聚类结果
    
    参数:
    X -- 降维后的数据
    labels -- 聚类标签或真实标签
    true_labels -- 真实标签（可选，用于对比）
    dimension -- 降维维度 (2或3)
    title -- 图表标题
    """
    if dimension not in [2, 3]:
        raise ValueError("维度必须是2或3")
    
    plt.figure(figsize=(10, 8))
    
    if dimension == 2:
        scatter = plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=5, alpha=0.7)
        plt.xlabel('Principal Component 1')
        plt.ylabel('Principal Component 2')
    else:
        ax = plt.axes(projection='3d')
        scatter = ax.scatter3D(X[:, 0], X[:, 1], X[:, 2], c=labels, cmap='tab10', s=5, alpha=0.7)
        ax.set_xlabel('Principal Component 1')
        ax.set_ylabel('Principal Component 2')
        ax.set_zlabel('Principal Component 3')
    
    plt.title(title)
    plt.colorbar(scatter, label='Cluster Label')
    plt.grid(True)
    plt.show()
    
    # 如果有真实标签，创建对比图表
    if true_labels is not None:
        plt.figure(figsize=(10, 8))
        
        if dimension == 2:
            plt.scatter(X[:, 0], X[:, 1], c=true_labels, cmap='tab10', s=5, alpha=0.7)
            plt.xlabel('Principal Component 1')
            plt.ylabel('Principal Component 2')
        else:
            ax = plt.axes(projection='3d')
            ax.scatter3D(X[:, 0], X[:, 1], X[:, 2], c=true_labels, cmap='tab10', s=5, alpha=0.7)
            ax.set_xlabel('Principal Component 1')
            ax.set_ylabel('Principal Component 2')
            ax.set_zlabel('Principal Component 3')
        
        plt.title(f"{title} (True Labels)")
        plt.colorbar(scatter, label='True Label')
        plt.grid(True)
        plt.show()

In [ ]:
# ====== 分层抽样函数 ======
def stratified_sampling(X, y, n_samples=1000):
    """
    分层抽样：从每个类别抽取相同数量的样本
    
    参数:
    X -- 输入数据
    y -- 标签
    n_samples -- 总抽样数量
    
    返回:
    X_sample -- 抽样后的数据
    y_sample -- 抽样后的标签
    """
    # 计算每个类别应抽取的样本数
    n_classes = len(np.unique(y))
    samples_per_class = n_samples // n_classes
    
    # 分层抽样
    sss = StratifiedShuffleSplit(n_splits=1, test_size=n_samples, random_state=42)
    for _, test_index in sss.split(X, y):
        X_sample = X[test_index]
        y_sample = y[test_index]
    
    return X_sample, y_sample

In [ ]:
# 3. 对降维后的数据进行K-Means聚类
# 对2D数据聚类
kmeans_2d = KMeans(n_clusters=10, n_init=10, random_state=42)
cluster_labels_2d = kmeans_2d.fit_predict(X_pca_manual_2d)
    
# 对3D数据聚类
kmeans_3d = KMeans(n_clusters=10, n_init=10, random_state=42)
cluster_labels_3d = kmeans_3d.fit_predict(X_pca_manual_3d)
    
# 4. 分层抽样获取1000个样本（每个类别100个）
print("\n进行分层抽样...")
X_sample_2d, y_sample_2d = stratified_sampling(X_pca_manual_2d, y, n_samples=1000)
X_sample_3d, y_sample_3d = stratified_sampling(X_pca_manual_3d, y, n_samples=1000)
    
# 获取抽样样本的聚类标签
cluster_labels_sample_2d = cluster_labels_2d[np.isin(X_pca_manual_2d, X_sample_2d).all(axis=1)]
cluster_labels_sample_3d = cluster_labels_3d[np.isin(X_pca_manual_3d, X_sample_3d).all(axis=1)]
    
# 5. 可视化结果
# 2D结果可视化（聚类结果 vs 真实标签）
print("\n可视化2D聚类结果...")
plot_cluster_results(
    X_sample_2d, 
    cluster_labels_sample_2d, 
    y_sample_2d,
    dimension=2, 
    title="2D PCA with K-Means Clustering (Stratified Sample)"
)
    
# 3D结果可视化（聚类结果 vs 真实标签）
print("\n可视化3D聚类结果...")
plot_cluster_results(
    X_sample_3d, 
    cluster_labels_sample_3d, 
    y_sample_3d,
    dimension=3, 
    title="3D PCA with K-Means Clustering (Stratified Sample)"
)

In [ ]:
# 6. 评估聚类效果（与真实标签的匹配程度）
# 使用调整兰德指数(Adjusted Rand Index)
from sklearn.metrics import adjusted_rand_score
    
ari_2d = adjusted_rand_score(y, cluster_labels_2d)
ari_3d = adjusted_rand_score(y, cluster_labels_3d)
    
print("\n聚类评估结果:")
print(f"2D降维聚类 - 调整兰德指数(ARI): {ari_2d:.4f}")
print(f"3D降维聚类 - 调整兰德指数(ARI): {ari_3d:.4f}")
    
# 解释方差比例可视化
# 计算所有主成分的解释方差比例
_, full_variance_ratio = manual_pca(X, n_components=X.shape[1])
cumulative_variance = np.cumsum(full_variance_ratio)
    
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance)+1), cumulative_variance, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
plt.axvline(x=2, color='g', linestyle='--', alpha=0.7)
plt.axvline(x=3, color='purple', linestyle='--', alpha=0.7)
plt.title('Cumulative Explained Variance by Principal Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.legend()
plt.show()